# Using Large Datasets with Pandas
Tutorial based on "Using large datasets with Pandas" by Miki Tebeka

In [1]:
# Mount Google Drive
#from google.colab import drive
#drive.mount('/content/drive')

# Load parquet or csv file
import pandas as pd
  
file_name = 'C:\\Users\\md82\\OneDrive - Anglia Ruskin University\\Documents\\mj-datalab\\edu-repo\\Python_BigData\\yellow_tripdata_2021-02.parquet'
df = pd.read_parquet(file_name)

# If not able to read parquet run the following code
#file_name = '/content/drive/MyDrive/BigData/yellow_trip_data.csv'
#df = pd.read_csv(file_name)

df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2021-02-01 00:40:47,2021-02-01 00:48:28,1.0,2.30,1.0,N,141,226,2,8.5,3.0,0.5,0.00,0.0,0.3,12.30,2.5,NaN
1,1,2021-02-01 00:07:44,2021-02-01 00:20:31,1.0,1.60,1.0,N,43,263,2,9.5,3.0,0.5,0.00,0.0,0.3,13.30,0.0,NaN
2,1,2021-02-01 00:59:36,2021-02-01 01:24:13,1.0,5.30,1.0,N,114,263,2,19.0,3.0,0.5,0.00,0.0,0.3,22.80,2.5,NaN
3,2,2021-02-01 00:03:26,2021-02-01 00:16:32,1.0,2.79,1.0,N,236,229,1,11.0,0.5,0.5,2.96,0.0,0.3,17.76,2.5,NaN
4,2,2021-02-01 00:20:20,2021-02-01 00:24:03,2.0,0.64,1.0,N,229,140,1,4.5,0.5,0.5,1.66,0.0,0.3,9.96,2.5,NaN


In [2]:
# Check size in memory of dataframe (MB)
mb = 1_000_000
df.memory_usage(deep=True).sum() / mb

282.308734

In [3]:
# Check size of the file (MB)
from pathlib import Path
Path(file_name).stat().st_size / mb

33.730552

In [4]:
# Calculate median distance by vendor
df.groupby('VendorID')['trip_distance'].median()

VendorID
1    1.60
2    1.74
Name: trip_distance, dtype: float64

In [5]:
# Avoid loading all the columns into memory - load only the data you need (in this case vendor and trip distance)
columns=['VendorID', 'trip_distance']
df = pd.read_parquet(file_name, columns=columns)
# df = pd.read_csv(file_name, usecols=['VendorID', 'trip_distance'])
df.memory_usage(deep=True).sum() / mb

32.592408

In [6]:
# Calculate again the median distance by vendor
df.groupby('VendorID')['trip_distance'].median()

VendorID
1    1.60
2    1.74
Name: trip_distance, dtype: float64

In [7]:
# Reload data
df = pd.read_parquet(file_name)
# df = pd.read_csv(file_name)

In [8]:
# Check data types
df.dtypes

VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                     float64
dtype: object

In [9]:
# Check for range of a specific variable (total amount in this case)
# Look for the min and max
df['total_amount'].describe()

count    1.358017e+06
mean     1.743016e+01
std      1.455326e+01
min     -6.348000e+02
25%      1.100000e+01
50%      1.415000e+01
75%      1.930000e+01
max      6.969300e+03
Name: total_amount, dtype: float64

In [10]:
# Check if this variable could be stored as a float32 type instead of float64
# to consume less memory
# Check for min and max values that can fit into float32
import numpy as np

np.finfo(np.float32)

finfo(resolution=1e-06, min=-3.4028235e+38, max=3.4028235e+38, dtype=float32)

In [11]:
# Calculate memory usage of column stored as float64
mb = 1_000_000
df['total_amount'].memory_usage(deep=True) / mb

21.728272

In [12]:
# Convert to float32 and check the size in memory - it's much less!
amount = df['total_amount'].astype(np.float32)
amount.memory_usage(deep=True) / mb

16.296204

In [13]:
# Create strings for categorical data (easier to read than indices)
names = {
    1: 'Creative',
    2: 'VeriFone',
}
df['vendor'] = df['VendorID'].map(names)

In [14]:
# Compare size in memory of indices versus string data
mb = 1_000_000
id_size = df['VendorID'].memory_usage(deep=True) / mb
name_size = df['vendor'].memory_usage(deep=True) / mb
print(f'id size: {id_size}, name size: {name_size}')

id size: 21.728272, name size: 99.135241


In [15]:
# Use data type 'category' to save memory
df['vendor'] = df['vendor'].astype('category')
df['vendor'].memory_usage(deep=True) / mb

12.222391

In [16]:
# It is easier to read than indices and uses less memory than strings
df['vendor'][:10]

0    Creative
1    Creative
2    Creative
3    VeriFone
4    VeriFone
5    VeriFone
6    Creative
7    VeriFone
8    Creative
9    Creative
Name: vendor, dtype: category
Categories (2, object): ['Creative', 'VeriFone']